# Phase 2P v2 - Leakage or typical images, with enough power to tell

**Run on:** Kaggle, free GPU. About 12.5 GPU hours in total, run as two shards.

---

### The question

Deduplication deletes the training images that have a test neighbour, and those
are the central members of dense clusters. Removing central images might cost
accuracy whether or not they leak anything. The control: delete the
near-duplicates of a **reference set R**, a held-out slice of training the same
size and class mix as the test fold, which is **never scored**. If the cost is
about typicality, both deletions cost the same. If it is about leakage, only the
deletion tied to the scored set costs more than chance.

### Design (fixed in v2)

Per fold, per replicate $j = 1..J$: draw R from the training partition,
stratified to the test fold's class counts. Pool $P$ = training minus R. Four
training sets, all from $P$, all scored on the identical test fold:

| condition | removed from P |
|---|---|
| test-dedup | images with cosine >= 0.98 to any **test** image |
| test-random | the same number, at random |
| ref-dedup | images with cosine >= 0.98 to any image in **R** |
| ref-random | the same number, at random |

v1 held R out only in the reference arms, so the two arms trained on different
amounts of data (about 390 against 515 images). Here every condition starts from
the same $P$.

### Power, computed before the run

From the archived v1 runs (ResNet50, about 400 to 525 training images):
retraining the same set under a new seed moves balanced accuracy by about 0.02
(sd); a new random deletion moves it by about 0.053. Taking 0.053 for every
condition, one replicate's excess has variance at most $4 \times 0.053^2 =
0.0112$. With $J = 40$, five folds, $\rho = n_\text{test}/n_\text{train} \approx
0.55$ and the Nadeau-Bengio correction, the 95% half-width is about **0.040**,
assuming the excess does not vary much between folds (if it does, the interval
is wider and more replicates do not fix it). Power at $\alpha = 0.05$: 0.97 for a
true excess of 0.076 (the v1 Phase 2N estimate), 0.57 for 0.041 (the v1 Phase 2P
estimate), 0.35 for 0.03. **So this run can detect an excess of the size v1 2N
suggested, and is not powered to detect one as small as 0.03.** We state that
now, not after.

### Analysis rule, fixed before the run

Per replicate, $x_{kj} = (\text{test-random} - \text{test-dedup}) -
(\text{ref-random} - \text{ref-dedup})$, in balanced accuracy. Per fold,
$\bar x_k$ = mean over $j$. Estimate: mean of $\bar x_k$ over the 5 folds, 95%
Nadeau-Bengio interval with $\rho$ = mean $n_\text{test}/n_\text{train}$ over the
runs, $t$ with 4 df.

- **Interval lower bound > 0**: leakage-specific excess detected. The paper may
  say deduplication costs more when the deleted images neighbour the scored set
  than when they neighbour an unscored set of the same size and class mix.
- **Interval upper bound < +0.03**: no leakage-specific excess larger than 3
  points. The typicality account is not excluded, and the paper says so.
- **Otherwise**: inconclusive. "Consistent with" stays, and the paper reports
  the interval.

The test-arm and reference-arm effects are also reported separately, with the
same interval. $J$, the threshold, the architecture and this rule do not change
after the run starts. The analysis cell computes nothing until all $5 \times J$
replicates are present.

## 1. Environment, data, similarity

In [ ]:
import subprocess, sys
for pkg in ["opencv-python-headless", "kagglehub", "scipy"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import os, json, time, shutil, random, gc, glob
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (precision_recall_fscore_support, accuracy_score,
                             balanced_accuracy_score)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("GPUs:", tf.config.list_physical_devices("GPU"))

IN_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working" if IN_KAGGLE else "/content"
RESULTS_DIR = f"{WORK}/fyp_phase2pr_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
ROWS_CSV = f"{RESULTS_DIR}/phase2pr_rows.csv"

N_FOLDS, THRESHOLD, J = 5, 0.98, 40
FOLDS_TO_RUN = [0, 1, 2, 3, 4]
MARGIN = 0.03
TIME_BUDGET_MIN = 11 * 60
T_START = time.time()
RESULTS = {"seed": SEED, "n_folds": N_FOLDS, "threshold": THRESHOLD, "J": J,
           "margin": MARGIN, "folds_to_run": FOLDS_TO_RUN, "tf": tf.__version__}

def save_json():
    with open(f"{RESULTS_DIR}/results.json", "w") as f:
        json.dump(RESULTS, f, indent=2, default=float)

In [ ]:
FOLDERS = {"Benign": "Bengin cases", "Malignant": "Malignant cases",
           "Normal": "Normal cases"}
DATA_ROOT = None
if os.path.isdir("/kaggle/input"):
    hits = [d for d, _, _ in os.walk("/kaggle/input")
            if os.path.basename(d) == "Malignant cases"]
    if hits:
        DATA_ROOT = os.path.dirname(hits[0])
if DATA_ROOT is None:
    import kagglehub
    DL = kagglehub.dataset_download("hamdallak/the-iqothnccd-lung-cancer-dataset")
    DATA_ROOT = os.path.dirname([d for d, _, _ in os.walk(DL)
                                 if os.path.basename(d) == "Malignant cases"][0])

SPLIT_URL = ("https://raw.githubusercontent.com/haseebkhan9081/"
             "iqothnccd-leakage-audit/main/split_seed42.csv")
subprocess.run(["wget", "-q", "-O", f"{WORK}/split_seed42.csv", SPLIT_URL], check=True)
df = pd.read_csv(f"{WORK}/split_seed42.csv")
df["path"] = [os.path.join(DATA_ROOT, FOLDERS[l], f)
              for l, f in zip(df["label"], df["file"])]
assert all(os.path.exists(p) for p in df["path"])
y_all = df["y"].to_numpy()

thumbs = []
for p in tqdm(df["path"], desc="thumbnails"):
    g = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2GRAY)
    g = cv2.resize(g, (64, 64), interpolation=cv2.INTER_AREA).astype(np.float32).ravel()
    g -= g.mean(); n = np.linalg.norm(g)
    thumbs.append(g / n if n > 0 else g)
T = np.stack(thumbs); S = T @ T.T; np.fill_diagonal(S, 0.0)
folds = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(df, y_all))
print("images", len(df))

## 2. The four training sets per replicate

In [ ]:
CONDS = ["test-dedup", "test-random", "ref-dedup", "ref-random"]

def replicate_sets(k, j):
    tr, te = folds[k]
    rng = np.random.default_rng([SEED, k, j])
    # R: class-stratified to the test fold's counts, drawn from training only
    te_counts = np.bincount(y_all[te], minlength=3)
    R = np.concatenate([rng.choice(tr[y_all[tr] == c], size=int(te_counts[c]),
                                   replace=False) for c in range(3)])
    P = np.setdiff1d(tr, R)
    hit_te = (S[np.ix_(P, te)] >= THRESHOLD).any(axis=1)
    hit_R = (S[np.ix_(P, R)] >= THRESHOLD).any(axis=1)
    n_te, n_R = int(hit_te.sum()), int(hit_R.sum())
    pick = lambda n, sub: np.sort(P[np.random.default_rng([SEED, k, j, sub])
                                    .choice(len(P), size=len(P) - n, replace=False)])
    sets = {"test-dedup": P[~hit_te], "test-random": pick(n_te, 1),
            "ref-dedup": P[~hit_R], "ref-random": pick(n_R, 2)}
    info = {"n_pool": int(len(P)), "removed_by_test": n_te, "removed_by_ref": n_R,
            "removed_by_both": int((hit_te & hit_R).sum())}
    return sets, info

stats_rows = []
for k in range(N_FOLDS):
    for j in range(J):
        sets, info = replicate_sets(k, j)
        te = folds[k][1]
        for c, s in sets.items():
            assert not set(s) & set(te), "test image in training"
        assert len(sets["test-dedup"]) == len(sets["test-random"])
        assert len(sets["ref-dedup"]) == len(sets["ref-random"])
        stats_rows.append({"fold": k, "rep": j, **info})
st = pd.DataFrame(stats_rows)
print(st.groupby("fold")[["n_pool", "removed_by_test", "removed_by_ref",
                          "removed_by_both"]].mean().round(1).to_string())
print("\nIf removed_by_test and removed_by_ref are close, both dedup conditions")
print("delete cluster-central images at comparable rates, which is the premise.")
RESULTS["removal_stats_by_fold"] = (st.groupby("fold").mean(numeric_only=True)
                                    .round(2).reset_index().to_dict("records"))
PLAN = [(k, j, c) for j in range(J) for k in FOLDS_TO_RUN for c in CONDS]
print("planned runs in this shard:", len(PLAN))
save_json()

## 3. Run (resumable)

In [ ]:
# Training runs in child processes of CHUNK runs each. A long Keras session on
# Kaggle died silently after 71 runs (8 Oct 2026, v1 of this notebook); a fresh
# process per chunk returns all memory, whatever the cause. The child trains
# exactly the training sets precomputed here; nothing about the design changes.
import pickle
SIZE, EPOCHS, BATCH, LR = 224, 30, 16, 1e-4
CHUNK = 30
X_PATH, JOBS_PATH = f"{WORK}/X_{SIZE}_uint8.npy", f"{WORK}/jobs.pkl"
X = np.empty((len(df), SIZE, SIZE, 3), np.uint8)   # same pixel values, 1/4 the memory
for i, p in enumerate(tqdm(df["path"], desc=f"load {SIZE}px")):
    X[i] = np.asarray(Image.open(p).convert("RGB").resize((SIZE, SIZE), Image.BILINEAR))
np.save(X_PATH, X); del X; gc.collect()

done = {}
for f in glob.glob("/kaggle/input/**/phase2pr_rows.csv", recursive=True) +          ([ROWS_CSV] if os.path.exists(ROWS_CSV) else []):
    for r in pd.read_csv(f).to_dict("records"):
        done[(int(r["fold"]), int(r["rep"]), r["condition"])] = r
print("runs carried over:", len(done))
if done:
    pd.DataFrame(list(done.values())).to_csv(ROWS_CSV, index=False)

jobs, cache = [], {}
for (k, j, c) in PLAN:
    if (k, j) not in cache:
        cache = {(k, j): replicate_sets(k, j)}
    sets, info = cache[(k, j)]
    jobs.append({"fold": k, "rep": j, "condition": c, "tr": sets[c], "te": folds[k][1],
                 "seed": SEED + 100000 + 1000 * k + 10 * j + CONDS.index(c), "info": info})
with open(JOBS_PATH, "wb") as fh:
    pickle.dump({"jobs": jobs, "y": y_all}, fh)

CHILD = r'''
import sys, os, gc, time, pickle, resource
import numpy as np, pandas as pd, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import (precision_recall_fscore_support, accuracy_score,
                             balanced_accuracy_score)
X_PATH, JOBS_PATH, ROWS_CSV, SIZE, EPOCHS, BATCH, LR, CHUNK = sys.argv[1:9]
SIZE, EPOCHS, BATCH, CHUNK, LR = int(SIZE), int(EPOCHS), int(BATCH), int(CHUNK), float(LR)
X = np.load(X_PATH, mmap_mode="r")
d = pickle.load(open(JOBS_PATH, "rb")); jobs, y_all = d["jobs"], d["y"]
done = set()
if os.path.exists(ROWS_CSV):
    for r in pd.read_csv(ROWS_CSV).to_dict("records"):
        done.add((int(r["fold"]), int(r["rep"]), r["condition"]))

def build():
    inp = layers.Input((SIZE, SIZE, 3))
    x = keras.Sequential([layers.RandomFlip("horizontal"),
                          layers.RandomRotation(0.05),
                          layers.RandomZoom(0.15, 0.15)], name="aug")(inp)
    base = keras.applications.ResNet50(include_top=False, weights="imagenet",
                                       input_shape=(SIZE, SIZE, 3))
    base.trainable = True
    for layer in base.layers[:-20]:
        layer.trainable = False
    x = base(keras.applications.resnet50.preprocess_input(x), training=False)
    x = layers.Dropout(0.3)(layers.GlobalAveragePooling2D()(x))
    return keras.Model(inp, layers.Dense(3, activation="softmax")(x))

n = 0
for jb in jobs:
    key = (jb["fold"], jb["rep"], jb["condition"])
    if key in done:
        continue
    if n >= CHUNK:
        break
    tr, te = jb["tr"], jb["te"]
    keras.backend.clear_session(); tf.random.set_seed(jb["seed"])
    counts = np.bincount(y_all[tr], minlength=3)
    cw = {i: float(len(tr) / (3 * c)) if c else 0.0 for i, c in enumerate(counts)}
    model = build()
    model.compile(optimizer=keras.optimizers.Adam(LR),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    t0 = time.time()
    model.fit(X[tr].astype(np.float32), y_all[tr], epochs=EPOCHS, batch_size=BATCH,
              class_weight=cw, verbose=0)
    pred = model.predict(X[te].astype(np.float32), verbose=0).argmax(1)
    row = {"fold": jb["fold"], "rep": jb["rep"], "condition": jb["condition"],
           "seed": jb["seed"], "n_train": int(len(tr)), "n_test": int(len(te)), **jb["info"],
           "accuracy": float(accuracy_score(y_all[te], pred)),
           "balanced_accuracy": float(balanced_accuracy_score(y_all[te], pred)),
           "macro_f1": float(precision_recall_fscore_support(
               y_all[te], pred, labels=[0, 1, 2], average="macro", zero_division=0)[2]),
           "minutes": (time.time() - t0) / 60,
           "max_rss_gb": resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6}
    pd.DataFrame([row]).to_csv(ROWS_CSV, mode="a", index=False,
                               header=not os.path.exists(ROWS_CSV))
    print(key, f"bal {row['balanced_accuracy']:.3f}", f"rss {row['max_rss_gb']:.1f} GB",
          flush=True)
    del model; gc.collect(); n += 1
'''
with open(f"{WORK}/train_chunk.py", "w") as fh:
    fh.write(CHILD)

def n_missing():
    have = set()
    if os.path.exists(ROWS_CSV):
        have = {(int(r["fold"]), int(r["rep"]), r["condition"])
                for r in pd.read_csv(ROWS_CSV).to_dict("records")}
    return sum((jb["fold"], jb["rep"], jb["condition"]) not in have for jb in jobs)

stalls = 0
while n_missing() and (time.time() - T_START) / 60 < TIME_BUDGET_MIN:
    before = n_missing()
    r = subprocess.run([sys.executable, f"{WORK}/train_chunk.py", X_PATH, JOBS_PATH,
                        ROWS_CSV, str(SIZE), str(EPOCHS), str(BATCH), str(LR), str(CHUNK)])
    after = n_missing()
    print(f"chunk exit {r.returncode}: {before - after} runs done, {after} left in shard",
          flush=True)
    stalls = stalls + 1 if after == before else 0
    if stalls >= 2:
        print("two chunks in a row made no progress; stopping."); break

done = {}
for r in pd.read_csv(ROWS_CSV).to_dict("records"):
    done[(int(r["fold"]), int(r["rep"]), r["condition"])] = r

FULL = [(k, j, c) for k in range(N_FOLDS) for j in range(J) for c in CONDS]
RESULTS["shard_complete"] = all(p in done for p in PLAN)
RESULTS["complete"] = all(p in done for p in FULL)
RESULTS["runs_present"] = sum(p in done for p in FULL)
print(f"{RESULTS['runs_present']} of {len(FULL)} runs present "
      f"(shard complete: {RESULTS['shard_complete']})")
save_json()

## 4. The answer (runs only when all five folds x J replicates are present)

In [ ]:
from scipy import stats

def nb_interval(d, rho):
    k = len(d)
    se = np.sqrt((1.0 / k + rho) * d.var(ddof=1))
    crit = stats.t.ppf(0.975, k - 1)
    p = float(2 * stats.t.sf(abs(d.mean() / se), k - 1)) if se > 0 else float("nan")
    return float(d.mean()), float(d.mean() - crit * se), float(d.mean() + crit * se), p

if not RESULTS["complete"]:
    print("Incomplete. No effects are computed until every replicate is present.")
else:
    cv = pd.DataFrame(list(done.values()))
    cv.to_csv(ROWS_CSV, index=False)
    w = cv.pivot_table(index=["fold", "rep"], columns="condition",
                       values="balanced_accuracy")
    w["eff_test"] = w["test-random"] - w["test-dedup"]
    w["eff_ref"] = w["ref-random"] - w["ref-dedup"]
    w["excess"] = w["eff_test"] - w["eff_ref"]
    per_fold = w.groupby("fold")[["eff_test", "eff_ref", "excess"]].mean()
    rho = float((cv.n_test / cv.n_train).mean())
    out = {"rho": rho, "J": J,
           "means": cv.groupby("condition").balanced_accuracy.mean().round(4).to_dict(),
           "per_fold": per_fold.round(4).reset_index().to_dict("records"),
           "within_fold_sd_of_excess": float(w.groupby("fold").excess.std().mean())}
    for name in ["eff_test", "eff_ref", "excess"]:
        m, lo, hi, p = nb_interval(per_fold[name].to_numpy(), rho)
        out[name] = {"mean": m, "ci": [lo, hi], "p": p}
        print(f"{name:9s} {m:+.4f}  95% CI ({lo:+.3f}, {hi:+.3f})  p={p:.4f}")
    lo, hi = out["excess"]["ci"]
    out["verdict"] = ("leakage-specific excess detected" if lo > 0 else
                      f"no leakage-specific excess larger than {MARGIN}" if hi < MARGIN
                      else "inconclusive")
    print("\nVERDICT (rule fixed before the run):", out["verdict"])
    RESULTS["verdict"] = out
    save_json()

    fig, ax = plt.subplots(figsize=(7, 4))
    for i, name in enumerate(["eff_test", "eff_ref", "excess"]):
        v = per_fold[name]
        ax.scatter([i] * len(v), v, alpha=0.5, s=26)
        m, (a, b) = out[name]["mean"], out[name]["ci"]
        ax.errorbar(i, m, yerr=[[m - a], [b - m]], fmt="o", color="k", capsize=6)
    ax.axhline(0, color="grey", lw=0.8)
    ax.set_xticks(range(3))
    ax.set_xticklabels(["test arm effect", "reference arm effect", "excess"])
    ax.set_ylabel("balanced accuracy"); ax.grid(axis="y", alpha=0.3)
    ax.set_title(f"Per-fold means over J={J} reference sets, 95% NB intervals",
                 fontsize=10)
    fig.tight_layout(); fig.savefig(f"{RESULTS_DIR}/fig_typicality_powered.png", dpi=200)

shutil.make_archive(f"{WORK}/fyp_phase2pr_results", "zip", RESULTS_DIR)
print("archived")